# 물 지수의 기준값을 바꾸면 무엇이 달라지는가?

스펙트럼 지수와 수면 판별의 차이 이해

**할 일** · 한강 주변 MNDWI 기준값 0·0.2의 면적과 장기 물 출현율 비교

위에서 아래로 실행. 코드를 바꾸기 전 결과를 예상하고, 실행 결과와 비교

## 실행 준비

CPU 런타임 사용. Earth Engine에 등록한 Cloud 프로젝트와 사용 권한 필요. [설정 안내](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md) 참고. 데이터 조회에는 본인 계정 인증 필요

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Earth Engine에 등록한 본인의 Cloud 프로젝트 ID
if not PROJECT_ID.strip():
    raise ValueError("PROJECT_ID에 본인의 등록된 Cloud 프로젝트 ID 입력")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
def mask_s2(image):
    scl = image.select('SCL')
    clear = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return (image.select(['B2', 'B3', 'B4', 'B8', 'B11'])
            .multiply(0.0001).updateMask(clear)
            .copyProperties(image, ['system:time_start']))
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi).filterDate('2020-05-01', '2020-10-01')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
              .map(mask_s2))

image = collection.median().clip(roi)
mndwi = image.expression('(g-s)/(g+s)', {'g':image.select('B3'),'s':image.select('B11')}).rename('MNDWI')
ndvi = image.expression('(n-r)/(n+r)', {'n':image.select('B8'),'r':image.select('B4')}).rename('NDVI')
water0 = mndwi.gt(0).rename('water0')
water02 = mndwi.gt(0.2).rename('water02')
areas = ee.Image.pixelArea().multiply(water0.addBands(water02)).divide(1e6).reduceRegion(ee.Reducer.sum(),roi,20,maxPixels=5_000_000)
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(roi)

In [ ]:
print('threshold areas (km2):', areas.getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi,10)
Map.addLayer(image, {'bands':['B4','B3','B2'],'min':0,'max':0.3}, 'RGB')
Map.addLayer(mndwi, {'min':-1,'max':1,'palette':['brown','white','blue']}, 'MNDWI')
Map.addLayer(water0.selfMask(), {'palette':['cyan']}, 'MNDWI > 0')
Map.addLayer(water02.selfMask(), {'palette':['blue']}, 'MNDWI > 0.2')
Map.addLayer(jrc, {'min':0,'max':100,'palette':['white','blue']}, 'JRC occurrence 1984–2021 (%)')
Map

기준값을 높였을 때 수면 후보 면적이 어떻게 달라지는지 설명하시오. RGB에서 구름·그림자·도시 표면과 혼동되는 곳을 찾으시오.

**해석 범위** · JRC 출현율은 1984–2021의 유효 관측 중 물이 관측된 비율. 2020 합성 영상의 현장 정답이 아님. 두 지도의 차이를 모두 오분류나 홍수로 단정하지 않음. B11은 원래 20 m 밴드이므로 20 m 계산 사용

## 자료와 해석 범위

- [Sentinel-2 SR](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)

- [JRC Global Surface Water 1.4](https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater)

관측 연도는 고정 사례의 기간. 수업 연도와 무관. 코드는 자료 처리 방법을 보여 주며 지도·수치는 인증 후 조회 결과로 판단